In [26]:
import re
import os
import pandas as pd



In [27]:
INPUT_FILE = "Survey.xlsx"
OUTPUT_FILE = "Survey_Clean.xlsx"

# Column names (after renaming)
ROLE_COL = "Which Title Best Fits your Current Role?"
INDUSTRY_COL = "What Industry do you work in?"
COUNTRY_COL = "Which Country do you live in?"
LANGUAGE_COL = "Favorite Programming Language"
PRIORITY_COL = "If you were to look for a new job today, what would be the most important thing to you?"
ETHNICITY_COL = "Ethnicity"

COLUMNS_TO_DROP = [
    "Time Taken (America/New_York)",
    "Browser",
    "OS",
    "City",
    "Country",
    "Referrer",
    "Time Spent",
]

INDUSTRY_MAP = {
    r"financ(e|al)": "Finance",
    r"manufact":     "Manufacturing",
    r"non\s*profit": "Nonprofit",
    r"consulti":     "Consulting",
}

# Order matters: the first matching pattern wins
ROLE_MAP = {
    r"data\s*engineer":                 "Data Engineer",
    r"software\s*(developer|engineer)": "Software Developer",
    r"\bai\b|machine learning|\bml\b":  "AI/ML Engineer",
    r"data\s*scien":                    "Data Scientist",
    r"data\s*analy":                    "Data Analyst",
    r"business\s*analy":                "Business Analyst",
    **INDUSTRY_MAP,
}

COUNTRY_MAP = {
    r"nigeri":       "Nigeria",
    r"argentin":     "Argentina",
    r"brazi":        "Brazil",
    r"^fin":         "Finland",
    r"^(ire|irel)$": "Ireland",
    r"^leba":        "Lebanon",
    r"perú":         "Peru",
    r"portug":       "Portugal",
    r"^uae$":        "United Arab Emirates",
    r"^uzb":         "Uzbekistan",
}

In [28]:
def load_survey(path=INPUT_FILE):
    """Read the survey and keep only the text after the last '-' in each column name."""
    df = pd.read_excel(path)
    df.columns = [col.split("-")[-1].strip() for col in df.columns]
    return df

starting_df = load_survey()


In [29]:
starting_df

,Unique ID,Email,Date Taken (America/New_York),Time Taken (America/New_York),Browser,OS,City,Country,Referrer,Time Spent,...,How Happy are you in your Current Position with the following? (Management),How Happy are you in your Current Position with the following? (Upward Mobility),How Happy are you in your Current Position with the following? (Learning New Things),How difficult was it for you to break into Data?,"If you were to look for a new job today, what would be the most important thing to you?",Male/Female?,Current Age,Which Country do you live in?,Highest Level of Education,Ethnicity
0,62a33b3db4da29969c62df3d,anonymous,6/10/2022,8:38,NaN,NaN,NaN,NaN,NaN,0:00:44,...,5.0,5.0,7.0,Very Difficult,Remote Work,Male,26,United States,NaN,White or Caucasian
1,62a33ba1bae91e4b8b82e35c,anonymous,6/10/2022,8:40,NaN,NaN,NaN,NaN,NaN,0:01:30,...,2.0,1.0,3.0,Very Difficult,Remote Work,Male,36,Canada,NaN,Asian or Asian American
2,62a33c2cbc6861bf3176bec1,anonymous,6/10/2022,8:42,NaN,NaN,NaN,NaN,NaN,0:02:18,...,5.0,7.0,7.0,Easy,Good Work/Life Balance,Male,23,Other (Please Specify):Nigeria,NaN,Black or African American
3,62a33c8624a26260273822f9,anonymous,6/10/2022,8:43,NaN,NaN,NaN,NaN,NaN,0:02:10,...,10.0,7.0,10.0,Very Easy,Remote Work,Female,35,Canada,NaN,White or Caucasian
4,62a33c91f3072dd892621e03,anonymous,6/10/2022,8:44,NaN,NaN,NaN,NaN,NaN,0:01:51,...,4.0,0.0,1.0,Difficult,Better Salary,Male,44,United States,NaN,Black or African American
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625,62b525563f28f20328aeee5c,anonymous,6/23/2022,22:45,NaN,NaN,NaN,NaN,NaN,0:00:50,...,6.0,5.0,7.0,Easy,Better Salary,Male,26,United States,Bachelors,White or Caucasian
626,62b5a3e29bc428d5345f6e89,anonymous,6/24/2022,7:45,NaN,NaN,NaN,NaN,NaN,0:03:12,...,7.0,4.0,6.0,Difficult,Better Salary,Male,21,United States,Bachelors,White or Caucasian
627,62b71083f31287f32e189026,anonymous,6/25/2022,9:41,NaN,NaN,NaN,NaN,NaN,0:04:43,...,2.0,1.0,2.0,Difficult,Better Salary,Male,26,Other (Please Specify):Oman,Bachelors,Asian or Asian American
628,62b795033b026e423f287ecd,anonymous,6/25/2022,19:06,NaN,NaN,NaN,NaN,NaN,0:02:17,...,6.0,4.0,3.0,Easy,Other (Please Specify):Career Advancedment,Male,24,Other (Please Specify):Costa Rica,Bachelors,Hispanic or Latino


In [30]:
def strip_other_prefix(val):
    """'Other (Please Specify): Data Engineer' -> 'data engineer'."""
    if pd.isna(val):
        return val
    return str(val).split(":")[-1].strip().lower()


In [31]:
def make_standardizer(mapping):
    """Return a function mapping free text to a standard label via regex,
    falling back to Title Case when nothing matches."""
    def standardize(val):
        if pd.isna(val) or val == "":
            return val
        for pattern, label in mapping.items():
            if re.search(pattern, val):
                return label
        return val.title()
    return standardize

In [32]:

def clean_and_standardize(series, mapping):
    return series.apply(strip_other_prefix).apply(make_standardizer(mapping))

# df = clean_and_standardize

In [33]:
def clean_language(val):
    if pd.isna(val):
        return val
    val = str(val).strip()
    if "sql" in val.lower():  # check SQL first so "PostgreSQL", "MySQL", etc. collapse to SQL
        return "SQL"
    return val.split(":")[0].strip().title()

In [34]:
def collapse_other(val):
    """Collapse any 'Other...' entry to 'Other'; leave other answers unchanged."""
    if pd.isna(val):
        return val
    val = str(val).strip()
    return "Other" if val.startswith("Other") else val


In [35]:
def clean_survey(df):
    df[ROLE_COL] = clean_and_standardize(df[ROLE_COL], ROLE_MAP)
    df[INDUSTRY_COL] = clean_and_standardize(df[INDUSTRY_COL], INDUSTRY_MAP)
    df[COUNTRY_COL] = clean_and_standardize(df[COUNTRY_COL], COUNTRY_MAP)
    df[LANGUAGE_COL] = df[LANGUAGE_COL].apply(clean_language)
    df[PRIORITY_COL] = df[PRIORITY_COL].apply(collapse_other)
    df[ETHNICITY_COL] = df[ETHNICITY_COL].apply(collapse_other)
    return df.drop(columns=COLUMNS_TO_DROP)


df = clean_survey(starting_df)


In [36]:
# show cleaned df
df

,Unique ID,Email,Date Taken (America/New_York),Which Title Best Fits your Current Role?,Did you switch careers into Data?,Current Yearly Salary (in USD),What Industry do you work in?,Favorite Programming Language,How Happy are you in your Current Position with the following? (Salary),How Happy are you in your Current Position with the following? (Work/Life Balance),...,How Happy are you in your Current Position with the following? (Management),How Happy are you in your Current Position with the following? (Upward Mobility),How Happy are you in your Current Position with the following? (Learning New Things),How difficult was it for you to break into Data?,"If you were to look for a new job today, what would be the most important thing to you?",Male/Female?,Current Age,Which Country do you live in?,Highest Level of Education,Ethnicity
0,62a33b3db4da29969c62df3d,anonymous,6/10/2022,Data Analyst,Yes,106k-125k,Healthcare,Python,9.0,9.0,...,5.0,5.0,7.0,Very Difficult,Remote Work,Male,26,United States,NaN,White or Caucasian
1,62a33ba1bae91e4b8b82e35c,anonymous,6/10/2022,Data Analyst,No,41k-65k,Finance,R,1.0,2.0,...,2.0,1.0,3.0,Very Difficult,Remote Work,Male,36,Canada,NaN,Asian or Asian American
2,62a33c2cbc6861bf3176bec1,anonymous,6/10/2022,Data Engineer,No,0-40k,Clean Energy,Python,0.0,8.0,...,5.0,7.0,7.0,Easy,Good Work/Life Balance,Male,23,Nigeria,NaN,Black or African American
3,62a33c8624a26260273822f9,anonymous,6/10/2022,Analytics Consultant,Yes,150k-225k,Finance,R,10.0,6.0,...,10.0,7.0,10.0,Very Easy,Remote Work,Female,35,Canada,NaN,White or Caucasian
4,62a33c91f3072dd892621e03,anonymous,6/10/2022,Data Analyst,Yes,41k-65k,Healthcare,R,1.0,4.0,...,4.0,0.0,1.0,Difficult,Better Salary,Male,44,United States,NaN,Black or African American
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625,62b525563f28f20328aeee5c,anonymous,6/23/2022,Data Analyst,Yes,125k-150k,Retail,R,8.0,5.0,...,6.0,5.0,7.0,Easy,Better Salary,Male,26,United States,Bachelors,White or Caucasian
626,62b5a3e29bc428d5345f6e89,anonymous,6/24/2022,Data Scientist,No,0-40k,Manufacturing,R,4.0,6.0,...,7.0,4.0,6.0,Difficult,Better Salary,Male,21,United States,Bachelors,White or Caucasian
627,62b71083f31287f32e189026,anonymous,6/25/2022,Student/Looking/None,Yes,0-40k,Tech,Python,1.0,2.0,...,2.0,1.0,2.0,Difficult,Better Salary,Male,26,Oman,Bachelors,Asian or Asian American
628,62b795033b026e423f287ecd,anonymous,6/25/2022,Data Engineer,No,0-40k,Tech,Python,6.0,6.0,...,6.0,4.0,3.0,Easy,Other,Male,24,Costa Rica,Bachelors,Hispanic or Latino


In [37]:

def main():
    df = clean_survey(load_survey())
    df.to_excel(OUTPUT_FILE, index=False)


In [38]:
if __name__ == "__main__":
    main()